In [ ]:
# import gzip
# import json
# import random
# 
# # Configuration
# input_file = ("ratings/Health_and_Household.csv.gz")
# output_file = "Health_and_Household_all.csv"
# sample_rate = 0.1  # 10%
# 
# def sample_dataset(input_path, output_path, rate):
#     count = 0
#     sampled_count = 0
#     
#     # Opening as 'rt' (read text) handles the decompression on-the-fly
#     with gzip.open(input_path, 'rt', encoding='utf-8') as f_in:
#         with open(output_path, 'w', encoding='utf-8') as f_out:
#             for line in f_in:
#                 count += 1
#                 # Randomly decide to keep this line
#                 if random.random() < rate:
#                     f_out.write(line)
#                     sampled_count += 1
#                 
#                 # Print progress every 100k lines
#                 if count % 100000 == 0:
#                     print(f"Processed {count} lines... Saved {sampled_count}")
# 
#     print(f"Finished! Total: {count} | Sampled: {sampled_count}")
# 
# sample_dataset(input_file, output_file, sample_rate)

Read Meta

In [1]:
import pandas as pd

# Load the filter
# sample_df = pd.read_csv("samples/Health_and_Household_10_percent.csv", header=None, names=['u', 'parent_asin', 'r', 't'])
# valid_asins = set(sample_df['parent_asin'].unique())

# Use the built-in Pandas JSON parser
# chunksize keeps memory low; lines=True handles the format
print("Reading with built-in JSON parser...")
chunks = pd.read_json("meta/FULL_meta_Health_and_Household.jsonl.gz", lines=True, chunksize=1000, compression='gzip')

full_meta = []
for i, chunk in enumerate(chunks):
    # Filter each chunk against your ASIN list
    filtered = chunk[chunk['categories'].apply(lambda x: 'Health Care' in x)]
    full_meta.append(filtered[['title','parent_asin','categories', 'rating_number', 'store']])
    if i%100 == 0:
        print(i)
# Combine into one final DataFrame
df_final = pd.concat(full_meta, ignore_index=True)

print(f"Success! Found {len(df_final)} matching products.")

Reading with built-in JSON parser...
0
100
200
300
400
500
600
700
Success! Found 186096 matching products.


In [ ]:
distinct_categories = (
    df_final['categories']
    .astype(str)
    .str.replace(r"^\[|\]$", "", regex=True) # Remove start and end brackets
    .str.split(",")                          # Split into lists
    .explode()                               # Flatten
    .str.strip()                             # Clean whitespace
    .unique()                                # Get distinct set
)

# Convert to a standard set for your lookup dictionary
# distinct_categories_set = set(distinct_categories)

In [ ]:
# 1. Clean and explode the categories
# 2. value_counts() gives the frequency of each unique entry
# 3. reset_index() converts the Series into a 2-column DataFrame
category_counts_df = (
    df_final['categories']
    .astype(str)
    .str.replace(r"^\[|\]$", "", regex=True) # Remove [ and ]
    .str.replace("'", "")                    # Remove quotes
    .str.split(",")                         # Split into list
    .explode()                              # Flatten
    .str.strip()                            # Clean whitespace
    .value_counts()                         # Count occurrences
    .reset_index()                          # Convert to DataFrame
)

# Rename columns for clarity
category_counts_df.columns = ['category', 'count']

# Filter out empty strings if any exist
category_counts_df = category_counts_df[category_counts_df['category'] != ""]

# Show the top results
print(category_counts_df.head(20))

In [0]:
exclude_household_cats = {
    # Cleaning & Maintenance
    'Household Cleaning', 'All-Purpose Cleaners', 'Bathroom Cleaners', 
    'Glass Cleaners', 'Kitchen Cleaners', 'Floor Cleaners', 'Carpet Cleaners',
    'Dishwashing', 'Dish soap', 'Dishwasher Detergent', 'Laundry', 
    'Laundry Detergent', 'Fabric Softener', 'Stain Removers', 'Bleach',
    'Drain Openers', 'Septic Treatments', 'Air Fresheners', 'Cleaning Tools', 
    'Sponges', 'Brooms', 'Mops & Bucket Sets', 'Squeegees', 'Dusting',
    'Trash Bags', 'Compost Bags', 'Lawn & Leaf Bags', 'Scouring Pads & Sticks',
    'Washing Machine Cleaners', 'Dishwasher Cleaners', 'Wood Polish', 'Household Supplies'
    
    # Paper & Plastic Disposables
    'Paper & Plastic', 'Toilet Paper', 'Paper Towels', 'Facial Tissues',
    'Napkins', 'Disposable Plates', 'Disposable Drinkware', 'Cups', 
    'Straws', 'Forks', 'Spoons', 'Aluminum Foil', 'Plastic Wrap', 
    'Food Storage Bags', 'Disposable Napkins', 'Disposable Food Storage',
    
    # General Household Utility & Hardware
    'Household Batteries', 'AA', 'AAA', '9V', 'C Batteries', 'D Batteries',
    'Coin & Button Cell', 'Lighters & Matches', 'Matches', 'Butane Fuel',
    'Pipe Cleaners', 'Charcoal', 'Hardware', 'Adhesives', 'Flints & Wicks',
    'Lighter Fluid', '9V Batteries', 'AA Batteries', 'Replacement Batteries',
    
    # Stationery & Gifts
    'Stationery & Gift Wrapping Supplies', 'Gift Wrap Paper', 'Gift Bags',
    'Gift Boxes', 'Greeting Cards', 'Notecard Sets', 'Notepads', 
    'Fine Writing Instruments', 'Labels & Rings', 'Gift Wrap Ribbons', 
    'Gift Wrap Bows', 'Gift Wrap Tags', 'Wrapping Tissue'
}

In [2]:
print(df_final.head())
# df_final = df_final[df_final['categories'].apply(lambda x: all(c.strip() not in exclude_household_cats for c in x))]
df_final = df_final.sort_values(by='rating_number', ascending=False).head(10000)

                                               title parent_asin  \
0                     Allegra Allergy 45ct + 15 Free  B00JENH5OI   
1  Rocky Mountain Oils Cinnamon Bark Essential Oi...  B07K363N3S   
2  Prevail Super Absorbent Underpads, Prevail Sup...  B00ACMDOOA   
3  DAWGS Women's Loudmouth Patterns Z Sandals | L...  B00VMGV2DK   
4  Analgesic Balm Counterpain Methyl Salicylate M...  B009NBVZGK   

                                          categories  rating_number  \
0  [Health & Household, Health Care, Over-the-Cou...             52   
1  [Health & Household, Health Care, Alternative ...            105   
2  [Health & Household, Health Care, Incontinence...              3   
3  [Health & Household, Health Care, Foot Health,...           5161   
4  [Health & Household, Health Care, Over-the-Cou...              4   

                 store  
0  Seven 'til Midnight  
1  Rocky Mountain Oils  
2        First Quality  
3                DAWGS  
4               Taisho  


In [12]:
len(df_final)

10000

In [55]:
df_final

,title,parent_asin,categories,rating_number,store
643,Gya Labs Pure Australian Tea Tree Oil for Skin...,B0BR4W8TBH,"[Health & Household, Health Care, Alternative ...",159102,Gya Labs
53327,Artizen 30ml Oils - Thuja Essential Oil - 1 Fl...,B0C632K498,"[Health & Household, Health Care, Alternative ...",130773,Artizen
7843,Essential Oils Set - Top 6 Organic Blends for ...,B06XRLR9RQ,"[Health & Household, Health Care, Alternative ...",126729,Lagunamoon
29984,"InnoGear Essential Oil Diffuser, Upgraded Diff...",B07YLKHR82,"[Health & Household, Health Care, Alternative ...",115979,InnoGear
14989,"MAJESTIC PURE Basil Essential Oil, Therapeutic...",B09NS3JKY6,"[Health & Household, Health Care, Alternative ...",106521,MAJESTIC PURE
...,...,...,...,...,...
38572,Silicone Foam Dressing with Gentle Adhesive Bo...,B0C6QLFJ8S,"[Health & Household, Health Care, First Aid, B...",993,vipflge
109145,EarTekPro High-Fidelity Concert Earplugs Reusa...,B07K167RV7,"[Health & Household, Health Care, Ear Care, Ea...",993,EarTekPro
4083,Attn: Grace Heavy Incontinence Pads for Women ...,B09D3HWL35,"[Health & Household, Health Care, Incontinence...",993,ATTN : GRACE
19934,Aluminium Keychain Medication Pill Box. Waterp...,B07N483T6G,"[Health & Household, Health Care, Over-the-Cou...",993,Deke Home


In [3]:
df_final['store'].value_counts()

store
Solimo             79
Dr. Scholl's       72
Boiron             69
Hyland's           68
Biofreeze          62
                   ..
Psoriasin           1
proVent             1
SPRINGSPIRIT        1
Easy@Home           1
Unique Wellness     1
Name: count, Length: 3665, dtype: int64

In [13]:
df_final.to_csv("FULL_meta_Health_and_Household_top10k.csv", sep='\x1f', index=False)

In [58]:
top_asins = df_final['parent_asin'].unique().tolist()
len(top_asins)

10000

In [10]:
import gzip
import pandas as pd

# 1. First, get the set of target ASINs from your Top 10k Metadata
# (Assuming 'top_10k_meta' is the DataFrame we created in the previous step)
target_asins = set(df_final['parent_asin'].unique())

input_file = "ratings/FULL_Health_and_Household.jsonl.gz"
output_file = "FULL_Health_and_Household_Top10k_Dense.csv"
import json
import gzip
import csv

def extract_to_clean_csv(input_path, output_path, target_set):
    count = 0
    saved_count = 0
    
    # These match your names=['user_id', 'parent_asin', 'rating', 'timestamp']
    fieldnames = ['user_id', 'parent_asin', 'rating', 'timestamp']
    
    print(f"Starting extraction for {len(target_set)} target products...")
    
    with gzip.open(input_path, 'rt', encoding='utf-8') as f_in:
        with open(output_path, 'w', encoding='utf-8', newline='') as f_out:
            # extrasaction='ignore' ensures the "text", "images", etc. are dropped
            writer = csv.DictWriter(f_out, fieldnames=fieldnames, extrasaction='ignore')
            
            # Note: I am NOT writing a header row so it stays compatible with your 
            # pd.read_csv(..., header=None) call.
            
            for line in f_in:
                count += 1
                try:
                    # Parse JSON once per line
                    data = json.loads(line)
                    
                    # Exact key match for the ASIN
                    if data.get('parent_asin') in target_set:
                        writer.writerow(data)
                        saved_count += 1
                except:
                    continue
                
                if count % 1000000 == 0:
                    print(f"Processed {count/1e6:.1f}M lines... Saved {saved_count} CSV rows")

    print(f"Finished! Extracted: {saved_count} rows into {output_path}")

# Run it
extract_to_clean_csv(input_file, output_file, target_asins)

Starting extraction for 10000 target products...
Processed 1.0M lines... Saved 119570 CSV rows
Processed 2.0M lines... Saved 247966 CSV rows
Processed 3.0M lines... Saved 375486 CSV rows
Processed 4.0M lines... Saved 507560 CSV rows
Processed 5.0M lines... Saved 645600 CSV rows
Processed 6.0M lines... Saved 764368 CSV rows
Processed 7.0M lines... Saved 887522 CSV rows
Processed 8.0M lines... Saved 1024232 CSV rows
Processed 9.0M lines... Saved 1152401 CSV rows
Processed 10.0M lines... Saved 1276289 CSV rows
Processed 11.0M lines... Saved 1401258 CSV rows
Processed 12.0M lines... Saved 1526384 CSV rows
Processed 13.0M lines... Saved 1660447 CSV rows
Processed 14.0M lines... Saved 1786575 CSV rows
Processed 15.0M lines... Saved 1910247 CSV rows
Processed 16.0M lines... Saved 2048521 CSV rows
Processed 17.0M lines... Saved 2187251 CSV rows
Processed 18.0M lines... Saved 2315459 CSV rows
Processed 19.0M lines... Saved 2446141 CSV rows
Processed 20.0M lines... Saved 2570286 CSV rows
Process

In [15]:
rev = pd.read_csv("../datasets/amazon/FULL_Health_and_Household_Top10k_Dense.csv")

In [17]:
rev.nunique()

AFKZENTNBQ7A7V7UXW5JJI6UGRYQ    2535881
B09RWQ64WD                         9996
4.0                                   5
1666386989060                   3176360
dtype: int64

In [ ]:
df_final.sort_values(by=['store'], ascending=False)

In [7]:
import json
import gzip
import csv

def extract_to_csv(input_path, output_path, target_set):
    count = 0
    saved_count = 0
    
    # Define the columns you actually need for your objective function
    fieldnames = ['user_id', 'parent_asin', 'rating', 'timestamp']
    
    print(f"Starting extraction for {len(target_set)} products...")
    
    with gzip.open(input_path, 'rt', encoding='utf-8') as f_in:
        with open(output_path, 'w', encoding='utf-8', newline='') as f_out:
            writer = csv.DictWriter(f_out, fieldnames=fieldnames, extrasaction='ignore')
            # Uncomment the next line if you want a header row in your CSV
            # writer.writeheader()
            
            for line in f_in:
                count += 1
                try:
                    data = json.loads(line)
                    asin = data.get('parent_asin')
                    
                    if asin in target_set:
                        # writer.writerow will only save the 4 columns in fieldnames
                        writer.writerow(data)
                        saved_count += 1
                except:
                    continue
                
                if count % 1000000 == 0:
                    print(f"Processed {count/1e6:.1f}M lines... Saved {saved_count} CSV rows")

    print(f"Finished! Saved {saved_count} rows to {output_path}")

# Run it
extract_to_csv(input_file, output_file, target_asins)

Starting extraction for 10000 target products...
Processed 1.0M lines... Saved 119570 dense reviews
Processed 2.0M lines... Saved 247966 dense reviews
Processed 3.0M lines... Saved 375486 dense reviews
Processed 4.0M lines... Saved 507560 dense reviews
Processed 5.0M lines... Saved 645600 dense reviews
Processed 6.0M lines... Saved 764368 dense reviews
Processed 7.0M lines... Saved 887522 dense reviews
Processed 8.0M lines... Saved 1024232 dense reviews
Processed 9.0M lines... Saved 1152401 dense reviews
Processed 10.0M lines... Saved 1276289 dense reviews
Processed 11.0M lines... Saved 1401258 dense reviews
Processed 12.0M lines... Saved 1526384 dense reviews
Processed 13.0M lines... Saved 1660447 dense reviews
Processed 14.0M lines... Saved 1786575 dense reviews
Processed 15.0M lines... Saved 1910247 dense reviews
Processed 16.0M lines... Saved 2048521 dense reviews
Processed 17.0M lines... Saved 2187251 dense reviews
Processed 18.0M lines... Saved 2315459 dense reviews
Processed 19.

In [5]:
import gzip

input_file = "ratings/FULL_Health_and_Household.jsonl.gz"

with gzip.open(input_file, 'rt', encoding='utf-8') as f:
    first_row = f.readline()
    print("--- RAW ROW DATA ---")
    print(repr(first_row)) 
    print("--- END ---")

--- RAW ROW DATA ---
'{"rating": 4.0, "title": "Just not for my purposes which is watercolor paint preservative", "text": "I wanted to love this bc it\\u2019s supposed to be great for preserving homemade watercolor paints, but my nose can\\u2019t get past the scent. (I suffer from debilitating migraines & horrific allergies so it\\u2019s a non-starter for me.). I had read it\\u2019s supposed to be very sweet & pleasant.  I found it to be very medicinal & pungent, but scent is a very subjective thing so don\\u2019t let me stop you.  I accidentally touched my finger to my mouth after opening the bottle & I will say it does seem like it will work for toothaches bc even after I washed my finger & again stuck my finger in my mouth the taste was still there. Lol.  Also, I skin tested on my arm to make sure there was no tea tree oil like other companies add when they sell theirs as \\u201cpure\\u201d. I\\u2019m allergic to tea tree oil.  It passed the skin test.  And yes, I do other stupid st